In [1]:
class Node():
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None

### **Queue** <br>
A queue Q is a data structure where elements are added at the back and removed at the front. <br>
Supported operations:
- Enqueue(x): add x to Q.<br>
- Dequeue(): remove and return the *first added* element in Q.<br>
- IsEmpty(): return True if Q is empty.<br><br>
All operations in O(1) time. Space is O(N). <br>
(This implementation uses a singly linked list)

In [12]:
class Queue():
    def __init__(self):
        self.size = 0
        self.front = None
        self.end = None
    
    def enqueue(self, x):
        node = Node(x)
        if self.end == None:
            self.front = node
            self.end = node
            self.size += 1
            return
        self.end.next = node
        self.end = node
        self.size += 1
    
    def dequeue(self):
        if self.is_empty():
            raise Exception("Dequeueing from an empty queue")
        remove = self.front
        self.front = self.front.next
        self.size -= 1
        if self.front == None:
            self.end = None
        return remove.value

    def is_empty(self):
        return self.size == 0

### **Defining the servers**

In [8]:
import random
import simpy
import numpy
from random import seed
import statistics

seed(42)

In [ ]:
class MM1server(object):
    def __init__(self, env):
        self.env = env
        self.server = simpy.Resource(env, capacity=1)
        self.arrival = numpy.random.poisson(lam=2, size=1000) # rate = lamdba [s-1]
        self.departure = numpy.random.poisson(lam=3, size=1000) # rate = mu [s-1]
        self.queue = Queue()

    # queue.enqueue(clock) to ensure that events are added in increasing time order

    def run(self):
        # Initial state
        clock = 0.0
        single_server = Queue()
        queue_server = Queue()
        server_busy = False

        while True:
            yield self.env.timeout(random.expovariate(self.arrival))
            self.queue.enqueue(self.env.now)
            if not self.server.count:  # if the server is idle
                self.env.process(self.serve())
    
    def serve(self, customer):
        yield self.env.timeout(random.expovariate(self.arrival))

In [17]:
n_events = 10

env = simpy.Environment()
car = MM1server(env, service_rate=3.0)
env.run(until=n_events)

In [ ]:
import heapq
import random

# ---- Parameters ----
lam = 2.0
mu = 3.0
SIM_TIME = 1000

# ---- State ----
clock = 0.0
queue = Queue()
server_busy = False

# ---- Stats ----
num_in_system = 0
total_delay = 0.0
num_served = 0
area_num_in_system = 0.0
last_event_time = 0.0

# ---- Events ----
event_list = []

def schedule_event(time, event_type):
    heapq.heappush(event_list, (time, event_type))

def exp(rate):
    return random.expovariate(rate)

schedule_event(exp(lam), "arrival")

# ---- Simulation ----
while clock < SIM_TIME:
    clock, event_type = heapq.heappop(event_list)

    # Time-average stats
    time_since_last = clock - last_event_time
    area_num_in_system += num_in_system * time_since_last
    last_event_time = clock

    if event_type == "arrival":
        num_in_system += 1

        # Schedule next arrival
        schedule_event(clock + exp(lam), "arrival")

        if server_busy:
            queue.enqueue(clock)
        else:
            server_busy = True
            service_time = exp(mu)
            schedule_event(clock + service_time, "departure")

    elif event_type == "departure":
        num_in_system -= 1
        num_served += 1

        if queue.size > 0:   # using size here
            arrival_time = queue.dequeue()
            delay = clock - arrival_time
            total_delay += delay

            service_time = exp(mu)
            schedule_event(clock + service_time, "departure")
        else:
            server_busy = False

# ---- Results ----
avg_delay = total_delay / num_served if num_served > 0 else 0
avg_num_in_system = area_num_in_system / clock

print(f"Average delay: {avg_delay:.4f}")
print(f"Average number in system: {avg_num_in_system:.4f}")
print(f"Utilization (rho): {lam / mu:.4f}")

Average delay: 0.6266
Average number in system: 1.8559
Utilization (rho): 0.6667
